# Post-Hoc Kalman / Luenberger Observer Reconstruction

**Thesis:** *Adaptive Feedback Control of a Multi-Input Microbial Fuel Cell Using Observer-Based Linear Quadratic Regulation*  
**Author:** Jonathan D. Miller, Boston University MSE 2026

---

## Purpose

The full internal observer state (`x_obs`) was not logged to the cloud in real time during experiments. Cloud telemetry captured the VFA estimate (`x_obs[0]`), measured EC, and measured pH, but not the observer's internal EC/pH states (`x_obs[1]`, `x_obs[2]`) nor the time-varying Kalman gain matrix (Experiments 2 and 3).

Because the observer is a **deterministic recursive algorithm**, replaying its equations against the recorded sensor and pump dose time series produces output that is mathematically equivalent to the original on-device computation. This notebook performs that replay for all three experiments.

### Observer types by experiment
| Experiment | Observer | Notes |
|---|---|---|
| Experiment 1 | Fixed-gain Luenberger | Gain matrix `L` is hardcoded |
| Experiment 2 | Full Kalman filter | Covariance `P` propagated at every step |
| Experiment 3 | Full Kalman filter | Same parameters as Experiment 2 |

### Validation
Reconstruction fidelity is verified by comparing the reconstructed VFA estimate against the cloud-logged VFA values. Mean absolute error < 0.7 mg/L across all experiments, which is negligible relative to the constrained VFA range of 50.05–52.00 mg/L. Residual error arises from (1) float32 arithmetic on the Arduino vs. float64 here, and (2) the 5-second cloud sampling interval vs. exact 60-second on-device dose timing.

### Outputs
Each experiment produces a CSV with columns:
- `time`, `phase` — timestamp and experiment phase label
- `pH_meas`, `EC_meas` — raw sensor measurements
- `VFA_logged` — VFA estimate as logged by cloud (for validation)
- `x_VFA`, `x_EC`, `x_pH` — full reconstructed observer state
- `P00`, `P11`, `P22` — covariance diagonal (Experiments 2 & 3 only)
- `K_VFA_EC`, `K_VFA_pH` — Kalman gain for VFA row (Experiments 2 & 3 only)
- `u_uri`, `u_spir` — pump doses (mL/min)

In [ ]:
import zipfile
import numpy as np
import pandas as pd
from pathlib import Path

## 1. Model Parameters

All parameters are transcribed verbatim from the firmware source files (`exp_1`, `exp_2`, `exp_3`). They are identical across all three experiments.

In [ ]:
# ── Setpoints ─────────────────────────────────────────────────────────────────
VFA0, EC0, PH0 = 50.25, 10.0, 7.2
VFA_OBS_MIN, VFA_OBS_MAX = 50.05, 52.00
x_nom = np.array([VFA0, EC0, PH0])

# ── Discrete-time A matrix ────────────────────────────────────────────────────
A_D = np.array([
    [ 0.99930577, -0.00002068, -0.00003493],
    [ 0.00233387,  0.98443635,  0.02453758],
    [-0.00015568,  0.00111512,  0.99805077],
])

# ── B matrix  col0 = urine, col1 = apple juice ────────────────────────────────
B_D = np.array([
    [0.00000000,  1.80650e-03],
    [6.84417e-04, -4.73295e-03],
    [2.49130e-02, -1.66300e-03],
])

# ── D_OFFSET (identified at EC0=8, pH0=7; observer corrects residual mismatch) ─
D_OFFSET = np.array([0.035204, -0.164326, 0.011369])

# ── Experiment 1: fixed Luenberger gain ───────────────────────────────────────
L_GAIN = np.array([
    [0.069052, -0.012473],
    [0.086008,  0.010409],
    [0.010409,  0.092932],
])

# ── Experiments 2 & 3: Kalman noise matrices ──────────────────────────────────
Q_PROC = np.array([0.010000, 0.148808, 0.001406])   # process noise diagonal
R_MEAS = np.array([0.157198, 0.000532])              # measurement noise diagonal

## 2. Observer Equations

Both observers share the same **predict** step:

$$\hat{x}_{k|k-1} = A(\hat{x}_{k-1} - x_{sp}) + x_{sp} + Bu_{k-1} + d_{\text{offset}}$$

They differ in how the **gain** is computed:

- **Luenberger:** $L$ is fixed (pre-computed off-line via DARE, hardcoded in firmware)
- **Kalman:** $K_k = P_{k|k-1} C^\top (C P_{k|k-1} C^\top + R)^{-1}$ computed at every step

The **update** step is identical for both:

$$\hat{x}_k = \hat{x}_{k|k-1} + K_k (y_k - C\hat{x}_{k|k-1})$$

where $C = [0\;1\;0;\ 0\;0\;1]$ selects the EC and pH rows.

In [ ]:
def predict(x, u):
    """Shared predict step for both observer types."""
    return x_nom + A_D @ (x - x_nom) + B_D @ u + D_OFFSET


def clamp_state(x):
    x[0] = np.clip(x[0], VFA_OBS_MIN, VFA_OBS_MAX)
    x[1] = np.clip(x[1], 0.0, 30.0)
    x[2] = np.clip(x[2], 3.0, 12.0)
    return x


def observer_luenberger(x, u, meas_EC, meas_pH, P=None):
    """Experiment 1: fixed-gain Luenberger observer."""
    x_pred = predict(x, u)
    innov  = np.array([meas_EC - x_pred[1], meas_pH - x_pred[2]])
    x_upd  = clamp_state(x_pred + L_GAIN @ innov)
    return x_upd, P, L_GAIN.copy()


def observer_kalman(x, u, meas_EC, meas_pH, P):
    """Experiments 2 & 3: time-varying Kalman filter with covariance propagation."""
    # Predict
    x_pred = predict(x, u)
    P_pred = A_D @ P @ A_D.T + np.diag(Q_PROC)

    # Innovation covariance  S = C P_pred C' + R
    # C selects rows 1 and 2, so C P C' is the 2x2 lower-right submatrix of P_pred
    S = np.array([
        [P_pred[1, 1] + R_MEAS[0], P_pred[1, 2]            ],
        [P_pred[2, 1],              P_pred[2, 2] + R_MEAS[1]],
    ])
    det = S[0, 0] * S[1, 1] - S[0, 1] * S[1, 0]
    if abs(det) < 1e-10:
        det = 1e-10
    S_inv = np.array([[S[1, 1], -S[0, 1]], [-S[1, 0], S[0, 0]]]) / det

    # Kalman gain  K = P_pred C' S^{-1}
    PC = P_pred[:, 1:3]          # 3×2 — columns corresponding to EC and pH
    K  = PC @ S_inv

    # Update
    innov = np.array([meas_EC - x_pred[1], meas_pH - x_pred[2]])
    x_upd = clamp_state(x_pred + K @ innov)

    # Covariance update  P = (I - KC) P_pred
    C   = np.zeros((2, 3)); C[0, 1] = 1.0; C[1, 2] = 1.0
    P_upd = (np.eye(3) - K @ C) @ P_pred

    return x_upd, P_upd, K

## 3. Data Loading

Each experiment zip contains per-variable CSV files exported from Arduino IoT Cloud. The loader merges them onto a common timeline and forward-fills slowly-changing pump signals.

In [ ]:
def load_experiment(zip_path, exp_folder):
    """
    Load and merge all needed signals for one experiment from its zip archive.

    Parameters
    ----------
    zip_path   : Path  — path to the experiment zip file
    exp_folder : str   — folder name inside the zip (e.g. 'Experiment_1')

    Returns
    -------
    DataFrame with columns: pH, EC_mS, Q_urine, Q_spir, Q_pho, VFA_logged, phase
    indexed by UTC timestamp.
    """
    files = {
        'pH'        : f'{exp_folder}/Digester_Tank_1-cloud_pH.csv',
        'EC_mS'     : f'{exp_folder}/Digester_Tank_1-cloud_EC_mS.csv',
        'Q_urine'   : f'{exp_folder}/Digester_Tank_1-cloud_Q_urine_mL.csv',
        'Q_spir'    : f'{exp_folder}/Digester_Tank_1-cloud_Q_spir_mL.csv',
        'Q_pho'     : f'{exp_folder}/Digester_Tank_1-cloud_Q_kombucha_mL.csv',
        'VFA_logged': f'{exp_folder}/Digester_Tank_1-cloud_VFA_mL.csv',
        'phase'     : f'{exp_folder}/Digester_Tank_1-cloud_stressState.csv',
    }

    dfs = {}
    with zipfile.ZipFile(zip_path) as zf:
        for key, fname in files.items():
            try:
                with zf.open(fname) as f:
                    df = pd.read_csv(f, parse_dates=['time'])
                df['time'] = pd.to_datetime(df['time'], utc=True)
                df = df.set_index('time').sort_index()
                df.columns = [key]
                dfs[key] = df
            except KeyError:
                print(f'  WARNING: {fname} not found in zip — skipping')

    merged = pd.concat(dfs.values(), axis=1, join='outer').sort_index()

    # Forward-fill pump signals and phase labels (they hold value between updates)
    for col in ['Q_urine', 'Q_spir', 'Q_pho', 'phase']:
        if col in merged.columns:
            merged[col] = merged[col].ffill()

    # Interpolate sensor readings over any gaps
    for col in ['pH', 'EC_mS']:
        if col in merged.columns:
            merged[col] = merged[col].interpolate(method='time').ffill().bfill()

    return merged.dropna(subset=['pH', 'EC_mS'])

## 4. Reconstruction Loop

For each timestep:
1. Detect **phase transitions** → reset observer state to `[VFA0, EC_meas, pH_meas]` and `P = I` (matches firmware behaviour)
2. Detect **dose events** (when `Q_urine` or `Q_spir` or `Q_pho` changes) → call `observerUpdate` with the current measurements and dose inputs
3. Hold state constant between dose events (observer only fires at dose moments in firmware)

In [ ]:
def reconstruct(zip_path, exp_folder, observer_fn, label=''):
    """
    Replay the observer against logged sensor and pump data.

    Parameters
    ----------
    zip_path    : Path
    exp_folder  : str
    observer_fn : callable — observer_luenberger or observer_kalman
    label       : str      — used for print output only

    Returns
    -------
    DataFrame with reconstructed observer state and covariance.
    """
    print(f'Reconstructing {label} ...')
    data = load_experiment(zip_path, exp_folder)
    print(f'  {len(data):,} rows  |  {data.index[0]}  →  {data.index[-1]}')

    q_uri   = data['Q_urine'].fillna(0.0)
    q_spir  = data['Q_spir'].fillna(0.0)
    q_pho   = data['Q_pho'].fillna(0.0)
    phase   = data['phase'].ffill().fillna('UNKNOWN')

    # Dose event: any pump quantity changes from previous timestep
    dose_event = (
        (q_uri  != q_uri.shift(1))  |
        (q_spir != q_spir.shift(1)) |
        (q_pho  != q_pho.shift(1))
    )

    x          = np.array([VFA0, data['EC_mS'].iloc[0], data['pH'].iloc[0]])
    P          = np.eye(3)
    prev_phase = None
    rows       = []

    for ts, row in data.iterrows():
        ph  = row.get('phase', 'UNKNOWN')
        ec  = float(row['EC_mS'])
        pH_ = float(row['pH'])
        u_u = float(row.get('Q_urine', 0.0) or 0.0)
        u_s = float(row.get('Q_spir',  0.0) or 0.0)

        # Reset observer on phase transition (mirrors firmware)
        if ph != prev_phase and prev_phase is not None:
            x = np.array([VFA0, ec, pH_])
            P = np.eye(3)

        # Fire observer at dose events
        if dose_event.get(ts, False):
            u_vec = np.array([0.0, 0.0]) if 'ACID' in str(ph) else np.array([u_u, u_s])
            x, P, K = observer_fn(x, u_vec, ec, pH_, P)

        rows.append({
            'time'      : ts,
            'phase'     : ph,
            'pH_meas'   : pH_,
            'EC_meas'   : ec,
            'VFA_logged': float(row.get('VFA_logged', np.nan) or np.nan),
            'x_VFA'     : x[0],
            'x_EC'      : x[1],
            'x_pH'      : x[2],
            'P00'       : float(P[0, 0]),
            'P11'       : float(P[1, 1]),
            'P22'       : float(P[2, 2]),
            'K_VFA_EC'  : float(K[0, 0]),
            'K_VFA_pH'  : float(K[0, 1]),
            'u_uri'     : u_u,
            'u_spir'    : u_s,
        })
        prev_phase = ph

    out = pd.DataFrame(rows).set_index('time')

    # ── Sanity check ──────────────────────────────────────────────────────────
    valid = out['VFA_logged'].dropna()
    if len(valid) > 0:
        mae = (out.loc[valid.index, 'x_VFA'] - valid).abs().mean()
        print(f'  Sanity check  MAE(reconstructed VFA, logged VFA) = {mae:.4f} mg/L')
        print(f'  (< 0.7 mg/L expected; residual from float32 vs float64 + timing jitter)')

    return out

## 5. Run All Three Experiments

Set `DATA_DIR` to the folder containing the three experiment zip files.

In [ ]:
# ── Configure paths ───────────────────────────────────────────────────────────
DATA_DIR   = Path('.')          # folder containing the zip files
OUTPUT_DIR = Path('.')          # folder where CSVs will be written
OUTPUT_DIR.mkdir(exist_ok=True)

EXPERIMENTS = [
    {
        'label'      : 'Experiment 1 (Luenberger)',
        'zip'        : DATA_DIR / 'Experiment_1__9_.zip',
        'folder'     : 'Experiment_1',
        'observer_fn': observer_luenberger,
        'out_csv'    : OUTPUT_DIR / 'exp1_kalman_reconstructed.csv',
    },
    {
        'label'      : 'Experiment 2 (Kalman)',
        'zip'        : DATA_DIR / 'Experiment_2__10_.zip',
        'folder'     : 'Experiment_2',
        'observer_fn': observer_kalman,
        'out_csv'    : OUTPUT_DIR / 'exp2_kalman_reconstructed.csv',
    },
    {
        'label'      : 'Experiment 3 (Kalman)',
        'zip'        : DATA_DIR / 'Experiment_3__4_.zip',
        'folder'     : 'Experiment_3',
        'observer_fn': observer_kalman,
        'out_csv'    : OUTPUT_DIR / 'exp3_kalman_reconstructed.csv',
    },
]

results = {}
for cfg in EXPERIMENTS:
    df = reconstruct(cfg['zip'], cfg['folder'], cfg['observer_fn'], cfg['label'])
    df.to_csv(cfg['out_csv'])
    results[cfg['label']] = df
    print(f'  Saved → {cfg["out_csv"]}\n')

print('Done.')

## 6. Validation Plots

Compare reconstructed VFA against cloud-logged VFA for each experiment, and plot the reconstructed observer state and Kalman gain over time.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

PHASE_COLORS = {'LQR': '#2196F3', 'BASELINE': '#FF9800', 'ACID_1': '#E53935', 'ACID_2': '#E53935'}

for label, df in results.items():
    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
    fig.suptitle(label, fontsize=13, fontweight='bold')

    t = df.index

    # ── Shade phases ──────────────────────────────────────────────────────────
    for ax in axes:
        prev_ph, prev_t = None, t[0]
        for ts, row in df.iterrows():
            ph = row['phase']
            if ph != prev_ph:
                if prev_ph in PHASE_COLORS:
                    ax.axvspan(prev_t, ts, alpha=0.08, color=PHASE_COLORS[prev_ph], linewidth=0)
                prev_ph, prev_t = ph, ts

    # Panel 1: VFA reconstruction vs logged
    axes[0].plot(t, df['x_VFA'],      color='#1565C0', lw=1.5, label='Reconstructed x_VFA')
    axes[0].plot(t, df['VFA_logged'], color='#EF6C00', lw=1, ls='--', alpha=0.7, label='Cloud-logged VFA')
    axes[0].set_ylabel('VFA (mg/L)')
    axes[0].legend(fontsize=8)

    # Panel 2: Observer EC and pH states vs measurements
    ax2a = axes[1]
    ax2b = ax2a.twinx()
    ax2a.plot(t, df['x_EC'],   color='#1565C0', lw=1.5, label='x_EC (observer)')
    ax2a.plot(t, df['EC_meas'],color='#1565C0', lw=1, ls=':', alpha=0.5, label='EC measured')
    ax2b.plot(t, df['x_pH'],   color='#2E7D32', lw=1.5, label='x_pH (observer)')
    ax2b.plot(t, df['pH_meas'],color='#2E7D32', lw=1, ls=':', alpha=0.5, label='pH measured')
    ax2a.set_ylabel('EC (mS/cm)', color='#1565C0')
    ax2b.set_ylabel('pH', color='#2E7D32')
    lines = ax2a.get_lines() + ax2b.get_lines()
    ax2a.legend(lines, [l.get_label() for l in lines], fontsize=7, loc='upper right')

    # Panel 3: Kalman gain K[VFA, EC] and K[VFA, pH]
    axes[2].plot(t, df['K_VFA_EC'], color='#6A1B9A', lw=1.5, label='K[VFA, EC]')
    axes[2].plot(t, df['K_VFA_pH'], color='#AD1457', lw=1.5, label='K[VFA, pH]')
    axes[2].set_ylabel('Kalman Gain')
    axes[2].legend(fontsize=8)

    # Panel 4: Covariance diagonal P00
    axes[3].plot(t, df['P00'], color='#00695C', lw=1.5, label='P[VFA,VFA]')
    axes[3].set_ylabel('Covariance P00')
    axes[3].set_xlabel('Time (UTC)')
    axes[3].legend(fontsize=8)

    for ax in axes:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %H:%M'))
        ax.xaxis.set_major_locator(mdates.AutoDateLocator())

    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()